# 04 Propagation and Lineage: Labels Surviving Transformations
**Repo:** local_contexts_geospatial  
**Author:** Lilly Jones, PhD, Daear Consulting, LLC  

## The Challenge
A TK label is only meaningful if it survives transformations.

In a real geospatial workflow, a dataset gets:
- Clipped to a study area
- Joined with other datasets
- Reprojected to a new CRS
- Resampled to a new resolution
- Aggregated to regional means
- Exported as a new file

At each step, there is an opportunity for the label to be silently
dropped, especially when a developer uses a standard function that
doesn't know about TK labels.

This notebook demonstrates:
1. **The drop scenario** what happens when labels are lost
2. **The correct pattern** `propagate_labels()` at each step
3. **The decorator** `@enforce_label_propagation` for function-level enforcement
4. **The merge problem** combining a labeled dataset with an unlabeled one
5. **`strip_labels()`** why it warns loudly and when it might be legitimate

In [ ]:
# Imports
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from localcontexts.labels import (
    TKLabel, TKMetadata, has_any_label, extract_tk_fields
)
from localcontexts.propagation import (
    propagate_labels,
    propagate_labels_strict,
    merge_labels,
    strip_labels,
    enforce_label_propagation,
    add_provenance_step,
)
from localcontexts.validation import (
    validate_provenance_intact,
    validate_label_present,
    ProvenanceError,
)

warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline

print("Setup complete.")

## The Drop Scenario
This is what happens when a developer is not thinking about label governance.

In [ ]:
# Source dataset with TK label
tk = TKMetadata(
    label     = TKLabel.NON_COMMERCIAL,
    community = "Oglala Lakota Nation",
    authority = "Tribal Data Governance Office",
    usage     = "Non-commercial environmental research only",
    contact   = "data@oglalalakota.org",
)

source_meta = tk.attach({
    "name":   "pine_ridge_ndvi_2012",
    "source": "MODIS MOD13Q1",
    "year":   2012,
})

print("Source metadata (has TK label):")
print(f"  has_any_label = {has_any_label(source_meta)}")
print(f"  tk:label      = {source_meta['tk:label']}")

In [ ]:
# The Drop Scenario: a developer creates output metadata from scratch

def clip_to_study_area_WRONG(data, source_meta):
    """
    BAD PATTERN: creates output metadata from scratch.
    The TK label is silently dropped.
    """
    # ... clipping logic would go here ...
    clipped_data = data

    # Developer creates new metadata but doesn't think about the label
    output_meta = {
        "name":    source_meta["name"] + "_clipped",
        "process": "clipped to study area",
    }
    # THE LABEL IS GONE. Nobody notices until someone uses it commercially.
    return clipped_data, output_meta


_, bad_output_meta = clip_to_study_area_WRONG(None, source_meta)

print("BAD output metadata (label dropped):")
print(f"  has_any_label = {has_any_label(bad_output_meta)}")
print(f"  Contents: {bad_output_meta}")
print()
print("The TK label is gone. Anyone who receives this output")
print("has no way to know it came from a labeled source.")

## The Correct Pattern
One line of code prevents the label from being dropped.

In [ ]:
def clip_to_study_area_CORRECT(data, source_meta):
    """
    GOOD PATTERN: propagate_labels() carries the TK label forward.
    """
    clipped_data = data

    output_meta = {
        "name":    source_meta["name"] + "_clipped",
        "process": "clipped to study area",
    }

    # One line: copy all tk: and bc: fields from source to output
    output_meta = propagate_labels(source_meta, output_meta)

    return clipped_data, output_meta


_, good_output_meta = clip_to_study_area_CORRECT(None, source_meta)

print("GOOD output metadata (label preserved):")
print(f"  has_any_label = {has_any_label(good_output_meta)}")
print(f"  tk:label      = {good_output_meta.get('tk:label')}")
print(f"  tk:community  = {good_output_meta.get('tk:community')}")
print()
print("The difference: one call to propagate_labels() carries")
print("all tk: and bc: fields from source to output automatically.")

In [ ]:
# Demonstrate a multi-step workflow where the label propagates through
# every transformation

print("Multi-step workflow with label propagation:")

# clip
step1_meta = propagate_labels(source_meta, {
    "name": "ndvi_2012_clipped", "process": "clip to Pine Ridge boundary"
})
step1_meta = add_provenance_step(step1_meta, "Clip to Pine Ridge boundary",
                                  workflow="04_propagation.ipynb")
print(f"After clip:      has_label={has_any_label(step1_meta)} "
      f"| tk:label={step1_meta.get('tk:label')}")

# reproject
step2_meta = propagate_labels(step1_meta, {
    "name": "ndvi_2012_projected", "process": "reproject to EPSG:5070"
})
step2_meta = add_provenance_step(step2_meta, "Reproject to EPSG:5070 (Albers Equal Area)")
print(f"After reproject: has_label={has_any_label(step2_meta)} "
      f"| tk:label={step2_meta.get('tk:label')}")

# Compute anomaly
step3_meta = propagate_labels(step2_meta, {
    "name": "ndvi_2012_anomaly", "process": "compute anomaly from 2000-2023 mean"
})
step3_meta = add_provenance_step(step3_meta, "Compute NDVI anomaly (departure from 2000-2023 mean)")
print(f"After anomaly:   has_label={has_any_label(step3_meta)} "
      f"| tk:label={step3_meta.get('tk:label')}")

print()
print("Provenance chain:")
chain = step3_meta.get("prov:chain", [])
for i, step in enumerate(chain, 1):
    print(f"  {i}. {step['process']}")

## The Decorator
For functions that are called many times in a pipeline, the
`@enforce_label_propagation` decorator automates propagation.

In [ ]:
@enforce_label_propagation(meta_arg="meta")
def compute_growing_season_mean(data, meta):
    """
    Compute growing season mean NDVI.
    The decorator automatically propagates TK labels from input meta
    to the returned output meta so no manual propagate_labels() call is needed.
    """
    # ... computate logic ...
    result_data = data
    output_meta = {
        "name":    meta.get("name", "") + "_gs_mean",
        "process": "growing season mean (May–Sep)",
    }
    return result_data, output_meta


result, result_meta = compute_growing_season_mean(None, source_meta)

print("Decorator result:")
print(f"  has_any_label = {has_any_label(result_meta)}")
print(f"  tk:label      = {result_meta.get('tk:label')}")
print(f"  process       = {result_meta.get('process')}")
print()
print("The decorator propagated the label without any explicit call.")
print("This is the safest pattern for pipeline functions.")

## The Merge Problem
When combining a labeled dataset with an unlabeled public dataset,
what label does the combined result carry?

In [ ]:
# Dataset A: Tribal-collected pasture condition scores (labeled)
meta_a = tk.attach({
    "name":   "pine_ridge_pasture_scores",
    "source": "Oglala Lakota range rider field observations",
})

# Dataset B: NOAA climate division PDSI using public federal data, no label
meta_b = {
    "name":   "sd_pdsi_monthly",
    "source": "NOAA Climate Division PDSI",
    "url":    "https://www.ncei.noaa.gov/pub/data/cirs/climdiv/",
}

print("Dataset A (labeled):")
print(f"  has_any_label = {has_any_label(meta_a)}")
print(f"  tk:label      = {meta_a.get('tk:label')}")
print()
print("Dataset B (unlabeled public data):")
print(f"  has_any_label = {has_any_label(meta_b)}")

# Merge: the combined dataset inherits the label from the labeled source
# propagate_labels_strict ensures the labeled source always wins
merged_meta = propagate_labels_strict(meta_a, {
    "name":    "pine_ridge_pasture_climate_merged",
    "sources": [meta_a["name"], meta_b["name"]],
    "process": "joined pasture scores with PDSI by month",
})

print()
print("Merged dataset:")
print(f"  has_any_label = {has_any_label(merged_meta)}")
print(f"  tk:label      = {merged_meta.get('tk:label')}")
print()
print("Rule: if ANY source carries a TK label, the combined dataset")
print("inherits that label. Public data does not dilute Tribal governance.")

## `strip_labels()` Warning and When It Applies

In [ ]:
# strip_labels() exists for transparency. It always emits a warning
# to make label removal visible in any workflow

import warnings as _w
# ensure warning shows even if previously seen
_w.simplefilter("always")   

print("Calling strip_labels() will warn:")
stripped = strip_labels(source_meta)

print()
print(f"After strip: has_any_label = {has_any_label(stripped)}")
print(f"Remaining keys: {list(stripped.keys())}")
print()
print("When is strip_labels() ever legitimate?")
print("  - Publishing only the aggregated result at a spatial scale")
print("    coarse enough that no individual Tribal data is identifiable")
print("  - When the community has explicitly authorized removal")
print("    for a specific output format or destination")
print("  - NEVER: as a convenience to avoid dealing with the label")
print()
print("The warning is intentional. Every use of strip_labels()")
print("should be documented in the workflow with a justification.")

## Provenance Check

In [ ]:
# validate_provenance_intact() catches accidental label drops
# Run it as an assertion after each transformation step

print("Provenance checks:")
print()

# Good: label is present
try:
    validate_provenance_intact(good_output_meta)
    print("good_output_meta: provenance intact")
except ProvenanceError as e:
    print(f"good_output_meta: {e}")

# Bad: label was dropped
try:
    validate_provenance_intact(bad_output_meta)
    print("bad_output_meta: provenance intact")
except ProvenanceError as e:
    print(f"bad_output_meta: {e}")

## Visual Summary: Drop vs. Propagate

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

steps = ["Source\ndataset", "After\nclip", "After\nreproject", "Final\noutput"]

# Bad path with label dropped
ax = axes[0]
ax.set_xlim(-0.5, len(steps) - 0.5)
ax.set_ylim(-0.5, 1.5)
ax.set_yticks([])
ax.set_xticks(range(len(steps)))
ax.set_xticklabels(steps, fontsize=9)
ax.set_title("WITHOUT propagate_labels()\nLabel is silently dropped",
             fontsize=10, fontweight="bold", color="#C0392B")

for i, (has_label, color) in enumerate([
    (True,  "#27AE60"),
    (False, "#C0392B"),
    (False, "#C0392B"),
    (False, "#C0392B"),
]):
    ax.add_patch(plt.Rectangle((i - 0.4, 0.1), 0.8, 0.8,
                                color=color, alpha=0.8, zorder=2))
    ax.text(i, 0.5, "TK\nlabel" if has_label else "NO\nlabel",
            ha="center", va="center", fontsize=9,
            color="white", fontweight="bold", zorder=3)
    if i < len(steps) - 1:
        ax.annotate("", xy=(i + 0.6, 0.5), xytext=(i + 0.4, 0.5),
                    arrowprops=dict(arrowstyle="->", color="black", lw=2))

ax.text(1, 1.3, "Label dropped here", ha="center", fontsize=9,
        color="#C0392B", fontweight="bold")
ax.set_facecolor("#FDFEFE")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

# Good path: label propagates through all steps
ax = axes[1]
ax.set_xlim(-0.5, len(steps) - 0.5)
ax.set_ylim(-0.5, 1.5)
ax.set_yticks([])
ax.set_xticks(range(len(steps)))
ax.set_xticklabels(steps, fontsize=9)
ax.set_title("WITH propagate_labels()\nLabel travels through every step",
             fontsize=10, fontweight="bold", color="#27AE60")

for i in range(len(steps)):
    ax.add_patch(plt.Rectangle((i - 0.4, 0.1), 0.8, 0.8,
                                color="#27AE60", alpha=0.8, zorder=2))
    ax.text(i, 0.5, "TK\nlabel",
            ha="center", va="center", fontsize=9,
            color="white", fontweight="bold", zorder=3)
    if i < len(steps) - 1:
        ax.annotate("", xy=(i + 0.6, 0.5), xytext=(i + 0.4, 0.5),
                    arrowprops=dict(arrowstyle="->", color="black", lw=2))

ax.text(1.5, 1.3, "Label propagates at each step", ha="center", fontsize=9,
        color="#27AE60", fontweight="bold")
ax.set_facecolor("#FDFEFE")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

plt.suptitle(
    "TK Label Propagation: Drop vs. The Correct Pattern",
    fontsize=12, fontweight="bold",
)
plt.tight_layout()
plt.show()

## Summary
| Function | When to use |
|---|---|
| `propagate_labels(src, dst)` | After every transformation, this is the default choice |
| `propagate_labels_strict(src, dst)` | When the labeled source must always win |
| `merge_labels(meta_a, meta_b)` | Combining two labeled sources |
| `@enforce_label_propagation` | On pipeline functions called many times |
| `strip_labels(meta)` | Only with explicit community authorization |
| `validate_provenance_intact(meta)` | As a post-step assertion in CI/testing |

**The rule:** If the input carries a TK or BC label, the output must too.
The only exception is explicit community authorization for a specific output.

## Next Notebook
**05 Spatial Authority:** How to assign labels based on geographic
extent and handle mixed-authority datasets where different parts
of the data carry different labels.